## ANTI-LAUNDERING MONEY

## Anti-Money Laundering (AML) Detection Pipeline
An end-to-end data science and machine learning pipeline engineered to detect money-laundering patterns across millions of financial records. 

The system processes the **IBM Transactions for Anti-Money Laundering (AML)** dataset (`HI-Small_Trans.csv`)—a benchmark dataset containing millions of financial transaction logs from September 2022.

---

## ## 1. Project Overview & Architecture

### 1. Data Ingestion & Import
* Import raw transaction logs containing attributes such as `Timestamp`, `From Bank`, `Account`, `To Bank`, `Account.1`, `Amount Paid`, `Payment Format`, and `Is Laundering`.

---

### 2. Feature Aggregation (SQL)
Extract behavioral indicators aggregated at the account level:
* **Max Amount Sent in 24 Hours:** Peak transaction amount sent within rolling 24-hour windows.
* **Volume Transferred through 7 Days:** Total transactional amount transferred over a 7-day rolling period.
* **Smurfing Flag:** Binary flag targeting transactions structured just below reporting thresholds ($9,000–$9,999).
* **Target Variable:** Aggregate ground-truth labels (`Is Laundering`) at the account level.

---

### 3. Network & Graph Analysis
Construct a directed transaction graph where accounts act as nodes and transactions form directed edges:
* Compute network metrics to identify pass-through accounts and circular transaction loops.
      * Betweenness centrality : How often a node lies on the shortest past between all other pairs .
      * Degree Centrality : How many direct connections a node has relative to the total number of connections in 
      the network
      * Clustering coefficient: Measures the tendance of a node's neighbors to be conneced to each other . 
   

---

### 4. Model & Validation
* Train a Machine Learning model (e.g., XGBoost / Logistic Regression) on the engineered features and evaluate performance using Precision-Recall metrics to detect fraudulent transactions.

Import Libraries

In [1]:
import duckdb
conn = duckdb.connect()
import pandas as pd
import numpy as np
from sklearn.metrics import precision_recall_curve


#### Data Ingestion 

In [2]:

conn = duckdb.connect()
CSV_PATH = r'C:\Users\Usuari\Downloads\HI-Small_Trans.csv'
conn.execute(f"""
    CREATE OR REPLACE TABLE txn AS
    SELECT * FROM read_csv_auto(
        '{CSV_PATH}',
        types={{'Timestamp': 'TIMESTAMP'}}
    )
""")

query_import = r"""
SELECT TIMESTAMP , "FROM BANK" , "Account" AS sender_account, "To Bank", "Account.1" AS receiver_account, 
"Amount Paid", "Payment Format","Is Laundering"
FROM 'C:\Users\Usuari\Downloads\HI-Small_Trans.csv'
LIMIT 5
"""

df = duckdb.query(query_import).df()
print (df.head())
del df 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            Timestamp From Bank sender_account To Bank receiver_account  \
0 2022-09-01 00:20:00       010      8000EBD30     010        8000EBD30   
1 2022-09-01 00:20:00     03208      8000F4580     001        8000F5340   
2 2022-09-01 00:00:00     03209      8000F4670   03209        8000F4670   
3 2022-09-01 00:02:00       012      8000F5030     012        8000F5030   
4 2022-09-01 00:06:00       010      8000F5200     010        8000F5200   

   Amount Paid Payment Format  Is Laundering  
0      3697.34   Reinvestment              0  
1         0.01         Cheque              0  
2     14675.57   Reinvestment              0  
3      2806.97   Reinvestment              0  
4     36682.97   Reinvestment              0  


### 2. Feature Aggregation (SQL)



### 3 Magnitude Feature Engineering (DuckDB SQL)
Two modular SQL qeuries are exected via SQB to extract , for both the receiver adn teh sender , the amount peak in a 24 hours window , the max of amount in a 7 days window , the smurfing events and the l abel is laundering

In [3]:
query_transfer= r"""
WITH selected_data AS (
    SELECT
    "Account" AS sender_account , 
    "Amount Paid" AS amount , 
    "Timestamp" , 
    "Is Laundering" As is_laundering,

    --SMURFING_FLAG
    CASE 
    when "Amount Paid" BETWEEN 9000 AND 10000 THEN 1 
    ELSE 0 
    END AS smurfing_flag,

    --24-hour rolling window 
    MAX("Amount Paid") OVER( PARTITION BY "Account" ORDER BY "Timestamp" 
    RANGE BETWEEN INTERVAL '24' HOUR  PRECEDING AND CURRENT ROW) 
    AS max_24_hour_window , 

    --7days Total Amount 
    SUM("Amount Paid") OVER (PARTITION BY "Account" ORDER BY "Timestamp" 
    RANGE BETWEEN INTERVAL '7' DAY PRECEDING AND CURRENT ROW ) 
    AS total_7_days_window, 

    FROM txn
    ) 
    SELECT sender_account , 
         MAX(amount) AS sender_peak_24h_amount,
         MAX(total_7_days_window) AS sender_peak_7d_volume , 
    SUM(smurfing_flag) AS sender_total_smurfing_events , 
    MAX(is_laundering) AS sender_is_laundering 
    FROM selected_data 
    GROUP BY sender_account 
    """

query_receiver = r"""
WITH selected_data AS (
    SELECT
    "Account.1" AS receiver_account , 
    "Amount Received" AS amount , 
    "Timestamp" , 
    "Is Laundering" As is_laundering,

    --SMURFING_FLAG
    CASE 
    when "Amount Received" BETWEEN 9000 AND 10000 THEN 1 
    ELSE 0 
    END AS smurfing_flag,

    --24-hour rolling window 
    MAX("Amount Received") OVER( PARTITION BY "Account.1" ORDER BY "Timestamp" 
    RANGE BETWEEN INTERVAL '24' HOUR  PRECEDING AND CURRENT ROW) 
    AS max_24_hour_window , 

    --7days Total Amount 
    SUM("Amount Received") OVER (PARTITION BY "Account.1" ORDER BY "Timestamp" 
    RANGE BETWEEN INTERVAL '7' DAY PRECEDING AND CURRENT ROW ) 
    AS total_7_days_window, 

    FROM txn
    ) 
    SELECT receiver_account , 
         
         MAX(amount) AS receiver_peak_24h_amount,
         MAX(total_7_days_window) AS receiver_peak_7d_volume , 
        SUM(smurfing_flag) AS receiver_total_smurfing_events , 
        MAX(is_laundering) AS receiver_is_laundering 
        FROM selected_data 
        GROUP BY receiver_account 
    """
    
df_aggregated_sender = conn.execute(query_transfer).df()
df_aggregated_receiver = conn.execute(query_receiver).df()
df_sender = df_aggregated_sender.rename(columns={
    'sender_account': 'account'})
df_receiver = df_aggregated_receiver.rename(columns={
    'receiver_account': 'account'})
                                        

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Merge the transfer and receiver accoutns aggregate statistics 
Realize an outer join to track  down either the Cash-Out and Source Accounts, and not removing any potential fraud transaction


In [4]:
df_tabular= df_sender.merge(df_receiver, on ='account' , how = 'outer') 

df_tabular['is_laundering'] = df_tabular[['sender_is_laundering','receiver_is_laundering']].max(axis=1)
df_tabular = df_tabular.drop( columns = ['sender_is_laundering' , 'receiver_is_laundering'])

## 4. Network & Graph Topology Analysis (NetworkX)
Once in possession of the aggregated dataframes for both sender and receiver accounts, the network graph can be constructed. The primary objective is to generate a visual map of the linkages between accounts, enabling the identification of cash-out accounts, origin accounts, and circular transactions that signal potential financial fraud.

- **Nodes:** Every distinct bank account represents a unique node.
- **Edges:** The financial transfers between accounts represent the directed edges of the graph.

First, a SQL query is executed to extract aggregate statistics of all financial transfers between each pair of accounts, forming the weighted edges. Subsequently, the graph is built using the NetworkX library.

Basic graph features are then computed to capture structural network topologies:

- **Centrality Metrics:**
  - **Degree Centrality:** Measures the number of direct connections a node has. In a directed flow network, **In-Degree** counts incoming edges , while **Out-Degree** counts outgoing edges (how many destinations funds are sent to, highlighting source behavior).
  - **PageRank:** Originally designed for web page ranking, it measures a node's overall importance by evaluating not just the quantity of its connections, but also the structural importance of the neighboring nodes connected to it. A connection originating from a heavily connected, influential node carries more weight. In a transaction network, it effectively highlights major hubs and critical bottlenecks through which high volumes of financial flow transit.
  - **Betweenness Centrality:**
    - Measures how often a node lies on the shortest path between all other pairs of nodes in the network.
    -  Nodes with high betweenness act as "bridges" or intermediaries. 
  - **Local Clustering and Density (Measuring Local Tightness):**
  These metrics look at the immediate neighborhood surrounding a node to see how interconnected its peers are.
  - **Clustering Coefficient:**
    - Measures the probability that the neighbors of a node are also connected to each other (forming triangles).
      
  - **Reciprocity:**
    -  Measures the proportion of mutual connections in a directed graph (if Node A points to Node B, does Node B also point back to Node A?).
    - Standard commercial transactions typically display low reciprocity (unidirectional payments from customer to merchant), whereas circular fund flows a frequently manifest distinct reciprocal behavioral patterns.
      

In [5]:
import networkx as nx
import matplotlib.pyplot as plt


In [6]:

#Define the edges 

edges = conn.execute(
r"""
SELECT "Account" AS sender_account, 
       "Account.1" AS receiver_account , 
       SUM("Amount Paid") AS total_amount , 
       COUNT(*) AS number_transfers ,
       MAX("Is Laundering") AS is_laundering
        FROM txn
       GROUP BY "Account" , "Account.1"               
"""
).df() 

G = nx.from_pandas_edgelist(
    edges, 
    source = 'sender_account' ,
    target = 'receiver_account' , 
    edge_attr = ['total_amount' , 'number_transfers','is_laundering'],
    create_using = nx.DiGraph()
)

del edges




### Compute Graph Metrics: Time Optimization 

When modeling a network graph from a dataset of 50 million transactions and 500,000 discrete bank accounts (nodes), evaluating metrics like clustering and betweenness centrality introduces prohibitive memory overhead that can  degrade system performance. To  prevent execution lag, runtime is monitorized using the built-in time module. The pipeline automatically skips specific feature calculations if the elapsed is above  a 10-minute threshold.

In [7]:
import random 
import time 

#Check the used time to compute  the clustering coefficient with 5000 nodes and scale too the total number of nodes
random.seed(42)
probe_nodes = random.sample(list(G.nodes), 5000)
t0 = time.time()
_ = nx.clustering(G, nodes=probe_nodes)           # real neighbors, not a subgraph view
probe_secs = time.time() - t0
est_full = probe_secs * (G.number_of_nodes() / 5000)


print(f"probe: {probe_secs:.2f}s for 5k nodes → est. {est_full/60:.1f} min for full graph")
in_centrality = nx.in_degree_centrality(G)
out_centrality =  nx.out_degree_centrality(G)
page_rank = nx.pagerank(G)
betweeness_centrality = nx.betweenness_centrality(G , k = 100 , seed = 42 , normalized = True)

# Compute local clustering and density 
if est_full < 600:                                # less than 10 min
    clustering_coefficient = nx.clustering(G)
else:
    print(f"⚠ SKIPPING clustering ({est_full/60:.0f} min est.) — too slow.")
    clustering_coefficient = {}                   # empty dict → NaN in df, filled with 0 later


df_graph_metrics= pd.concat ([pd.Series(in_centrality, name = 'In-Degree' ), pd.Series(out_centrality, name = 'Out-Degree')  , 
                            pd.Series(page_rank, name = 'PageRank') ,pd.Series(betweeness_centrality ,name = 'Betweeness'),   
                            pd.Series(clustering_coefficient, name = 'Clustering')],
                              axis = 1).reset_index().rename(columns = {'index':'account'})



df_final = df_tabular.merge(df_graph_metrics, on = 'account' , how = 'outer').fillna(0)


probe: 0.93s for 5k nodes → est. 1.6 min for full graph


 ## 5. Machine Learning Pipeline & Compliance Tuning (XGBoost)

Create a full pipeline and train a Machine Learning Model to flag laundered accounts and handle hard imbalances between artifical laundering and legitimate transactions

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report , average_precision_score


from xgboost import XGBClassifier

In [9]:

#Remove the target columnd and high-cardinal column 'account'
feature_col = [ col for col in df_final.columns if col not in ['account' , 'is_laundering']]
X = df_final[feature_col]
y = df_final['is_laundering']

X_train , X_test , y_train , y_test = train_test_split(X, y , train_size = 0.8 , random_state = 42 ,stratify= y)

n_pos = max((y_train == 1).sum(), 1)
n_neg = (y_train == 0).sum()
scale_pos_weight = n_neg / n_pos
params = {  
    'objective' : 'binary:logistic',
    'max_depth' : 4,
    'learning_rate' : 0.1 , 
    'n_estimators' : 100 , 
    'alpha' : 0.0001 , 
    'scale_pos_weight': scale_pos_weight,
    'eval_metric': 'aucpr'
}

model = XGBClassifier(**params)

model.fit(X_train , y_train)

y_predict = model.predict(X_test)
y_predict_probs = model.predict_proba(X_test)[:,1] #Slicing, take the second column 

precision , recall, thresholds = precision_recall_curve(y_test, y_predict_probs) 

print(f"PR-AUC: {average_precision_score(y_test, y_predict_probs):.4f}")

# ----- THRESHOLD SEARCH : HIGHEST threshold that still hits 95% recall ---

target_recall = 0.95 
valid_indices = np.where(recall>=target_recall)[0]
if len(valid_indices) == 0:
    raise RuntimeError(f"Recall {target_recall} unachievable")
best_idx = valid_indices[-1]  #The array are sorted by increasing thresholds. We pich the lowest , to agressively dectect laundering 
best_threshold = thresholds[best_idx]
achieved_precision = precision[best_idx]

custom_preds = (y_predict_probs >= best_threshold).astype(int)
print(f"\n Target recall : {target_recall * 100}%")
print(f"  Required threshold : {best_threshold:.4f}%")
print(f"  Resulting precision at that threshold : {achieved_precision:.4f}")
print("\n", classification_report(y_test, custom_preds, digits=4, zero_division=0))

# 4. Apply this custom threshold to your test predictions



PR-AUC: 0.0731

 Target recall : 95.0%
  Required threshold : 0.3293%
  Resulting precision at that threshold : 0.0251

               precision    recall  f1-score   support

         0.0     0.9989    0.5384    0.6996    101745
         1.0     0.0251    0.9504    0.0489      1271

    accuracy                         0.5435    103016
   macro avg     0.5120    0.7444    0.3743    103016
weighted avg     0.9868    0.5435    0.6916    103016



The model achieves a PR-AUC of 0.0742 on the test set. Contrasting against the random baseline for this dataset (the positive class prevalence of 1.23%), the model delivers a lift of approximately 6x over random guessing. While the pipeline outperforms chance, the PR-AUC remains below the typical 0.18–0.25 target range for advanced AML datasets.

**Diagnosis** :  Why Feature Engineering Must Change
 Inspection of feature importances and class-conditional distributions reveals that the model's discriminative power is based in magnitude-related features — account transaction volume, degree centrality, and PageRank — rather than in behavioral features tha characterize laundering rirks.

## New Features 

To train the model with meaningful indicators, we shifted away from simple magnitude metrics toward **behavioral features** that capture unusual movement patterns: rapid spikes in transaction frequencies, receiving numerous small transfers from distinct counterparties, or high-velocity pass-through activity (potential mule accounts).* **Fan-In & Fan-Out Ratios:**
  * `fan_in`: Count of unique senders delivering funds (`IN`). Identifies  collection behavior.
  * `fan_out`: Count of unique receivers accepting funds (`OUT`). Identifies distribution behavior.

* **Dwell Times:**
  * `min_dwell_time` & `avg_dwell_time`: Measure the minimum and average time (in seconds) funds remain in an account before being moved out (`IN` immediately followed by `OUT`). Identifies automated layering accounts.

* **Rolling 1-Hour Burst (`max_hourly_burst`):**
  * Measures the maximum number of transactions an account executed within any sliding 1-hour window. Captures rapid bursts of activity.

## ## 3. Behavioral Feature Engineering (DuckDB SQL) [Again]

Two modular SQL queries are executed  via DuckDb to extract  behavorial features : Fan-IN &Fan-Out Rations, Dwell times in the first qeury , followed by  the balance_drain rate 1-hour transaction peaks  in the second query. 

Note that's not the most efficient way, since it forces the compiler to process the full dataset twice. A single unified query would  reduce memory usage, but difficulting  the readiness of the full query.

In [12]:
query_core = """
WITH all_transferences AS ( 
-- Select all transferences, both the receiver and sender, with direction variable distinguishinh them
SELECT  "Account" AS account,
        "Timestamp",
        "Account.1" AS counterparty , 
        "Amount Paid" AS amount,
        'OUT' AS direction 
FROM txn
UNION ALL 
SELECT  "Account.1" AS account , 
        "TimeStamp",
        "Account" AS counterparty,
        "Amount Received" AS amount , 
         'IN' AS direction 
FROM txn ),
calculated_deltas AS ( 
--Find the time and direction of all  following transference
SELECT account ,
       Timestamp , 
       direction , 
       counterparty, 
       LEAD("Timestamp") OVER ( PARTITION BY account ORDER BY "Timestamp") AS next_time ,
       LEAD(direction) OVER ( PARTITION BY account ORDER BY "Timestamp") AS next_direction 
       FROM all_transferences
) 

SELECT 
--FIND the dwell time between one transference in and one transference out 
account , 
MIN(date_diff('second' , 
                Timestamp, 
                next_time))
                AS min_dwell_time,
                AVG(date_diff(
                'second' , 
                Timestamp, 
                next_time))
                AS avg_dwell_time,
-- Count the number of incoming transactions(fan_in) and the number of outgoing transactions(fan-out)
COUNT(DISTINCT CASE WHEN direction = 'IN' THEN counterparty END) AS fan_in_ratio, 
COUNT(DISTINCT CASE WHEN direction = 'OUT' THEN counterparty END) AS fan_out_ratio, 
FROM calculated_deltas
WHERE direction = 'IN' AND next_direction = 'OUT' 
GROUP BY account
"""


query_burst = """
-- CTE, the same than before
WITH all_transferences AS ( 
    SELECT 
        "Account" AS account,
        "Timestamp",
        "Amount Paid" AS amount,
        'OUT' AS direction 
    FROM txn
    UNION ALL 
    SELECT 
        "Account.1" AS account, 
        "Timestamp",
        "Amount Received" AS amount, 
        'IN' AS direction 
    FROM txn 
),
-- CTE rolling counts ,querying all the transferences within a 1-hours window 
rolling_counts AS ( 
    SELECT account,
           Timestamp,
           amount,
           direction,
           COUNT(*)
           OVER ( PARTITION BY account 
           ORDER BY Timestamp 
           RANGE BETWEEN INTERVAL '1 hour' PRECEDING AND CURRENT ROW)
           AS rolling_window_1_hour
           FROM all_transferences)
    SELECT account , 
            MAX(rolling_window_1_hour) AS peak_rolling_window,
            SUM(CASE WHEN direction = 'OUT' then amount ELSE 0 END) / NULLIF(SUM(CASE WHEN direction='IN' THEN amount ELSE 0 END) ,0) 
            AS balance_drain_rate 
            FROM rolling_counts
            GROUP BY account
            """






In [13]:
df_core = conn.execute(query_core).df()
df_burst = conn.execute(query_burst).df()



    

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [14]:
df_features = pd.merge(df_core, df_burst , on = 'account' , how = 'left')
df_master = pd.merge(df_features, df_graph_metrics, on="account", how="outer")
df_labels = conn.execute("""
   SELECT account, MAX(CAST(is_laundering AS INTEGER)) AS is_laundering 
    FROM (
        SELECT "Account" AS account, "Is Laundering" AS is_laundering FROM txn
        UNION ALL
        SELECT "Account.1" AS account, "Is Laundering" AS is_laundering FROM txn
    ) 
    GROUP BY account
""").df()

df_ml = pd.merge(df_master, df_labels, on="account", how="left")
df_ml["is_laundering"] = df_ml["is_laundering"].fillna(0)


## 4. Machine Learning Pipeline & Compliance Tuning (XGBoost)[Again]

Again, a XgBoost Machine Learniag Model is created to detect laundered accounts. Now, with the improved behavioral features, we expect an increase of the PR-AUC score , reflect ana better capacity to distinguish legitimiate and launderer accounts .

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    precision_recall_curve,
)


In [16]:

# 1. Feature Selection (Fixed .cols to .columns)
feature_cols = [
    col for col in df_ml.columns if col not in ["account", "is_laundering"]
]
X = df_ml[feature_cols]
y = df_ml["is_laundering"]

# 2. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Handle AML Class Imbalance
n_pos = max((y_train == 1).sum(), 1)
n_neg = (y_train == 0).sum()
scale_pos_weight = n_neg / n_pos

params = {
    "objective": "binary:logistic",
    "max_depth": 4,
    "learning_rate": 0.1,
    "n_estimators": 100,
    "alpha": 0.0001,
    "scale_pos_weight": scale_pos_weight,
    "eval_metric": "aucpr",
}

# 4. Model Training (XGBoost handles missing structural values natively)
model = XGBClassifier(**params)
model.fit(X_train, y_train)

# 5. Evaluation & Probability Scoring
y_predict_probs = model.predict_proba(X_test)[:, 1]
print(f"PR-AUC: {average_precision_score(y_test, y_predict_probs):.4f}")

# 6. Compliance Threshold Tuning (Targeting 95% Recall)
precision, recall, thresholds = precision_recall_curve(y_test, y_predict_probs)

target_recall = 0.95
valid_indices = np.where(recall >= target_recall)[0]
if len(valid_indices) == 0:
  raise RuntimeError(f"Recall {target_recall} unachievable")

best_idx = (
    valid_indices[-1]
)  # Lowest threshold that satisfies the 95% recall target
best_threshold = thresholds[best_idx]
achieved_precision = precision[best_idx]

custom_preds = (y_predict_probs >= best_threshold).astype(int)
print(f"\nTarget Recall: {target_recall * 100}%")
print(f"Required Threshold: {best_threshold:.4f}")
print(f"Resulting Precision: {achieved_precision:.4f}")
print("\nClassification Report at Target Threshold:")
print(classification_report(y_test, custom_preds, digits=4, zero_division=0))

PR-AUC: 0.0829

Target Recall: 95.0%
Required Threshold: 0.3230
Resulting Precision: 0.0245

Classification Report at Target Threshold:
              precision    recall  f1-score   support

           0     0.9988    0.5266    0.6897    101745
           1     0.0245    0.9512    0.0477      1271

    accuracy                         0.5319    103016
   macro avg     0.5117    0.7389    0.3687    103016
weighted avg     0.9868    0.5319    0.6817    103016



# 6. Results, Diagnosis & Performance Evaluation

There has been an incremental rise in the PR-AUC score from the previous iteration. However, achieving high recall in a severely imbalanced dataset forces the decision threshold down significantly to flag potential launderers, which compromises precision. A precision of 0.0245 indicates that roughly 98% of the accounts classified as fraudulent are actually legitimate. Therefore, this model is still not production-ready, even while successfully capturing 95% of potential launderers.